In [ ]:
import pandas as pd
import os
import requests
from PIL import Image
from tqdm import tqdm
import shutil
import random


In [ ]:
def load_and_filter_csv(csv_path='snakes.csv', min_samples=200):
    df = pd.read_csv(csv_path, low_memory=False)

    # Оставляем только нужные столбцы
    df = df[['scientific_name', 'common_name', 'image_url', 'quality_grade']]

    # Только research-grade
    df = df[df['quality_grade'] == 'research']

    # Убираем строки без изображений
    df = df[df['image_url'].notna()]

    # Фильтрация видов по количеству
    counts = df['scientific_name'].value_counts()
    valid_species = counts[counts >= min_samples].index

    df = df[df['scientific_name'].isin(valid_species)].copy()

    print(f"✔ Осталось видов: {df['scientific_name'].nunique()}")
    print(f"✔ Всего изображений: {len(df)}")

    return df


In [ ]:
def download_images(df, out_dir='dataset_raw', max_per_species=5000):
    os.makedirs(out_dir, exist_ok=True)

    for species, group in tqdm(df.groupby('scientific_name'), desc='Виды'):
        species_dir = os.path.join(out_dir, species.replace(' ', '_'))
        os.makedirs(species_dir, exist_ok=True)

        group = group.sample(
            min(len(group), max_per_species),
            random_state=42
        )

        for i, row in group.iterrows():
            try:
                url = row['image_url']
                if url.startswith('//'):
                    url = 'https:' + url

                file_path = os.path.join(species_dir, f'{i}.jpg')
                if os.path.exists(file_path):
                    continue

                r = requests.get(url, timeout=10)
                if r.status_code == 200:
                    with open(file_path, 'wb') as f:
                        f.write(r.content)

                    # Проверка валидности
                    Image.open(file_path).verify()

            except:
                if os.path.exists(file_path):
                    os.remove(file_path)


In [ ]:
def split_dataset(
    src='dataset_raw',
    dst='dataset_final',
    target=500,
    ratios=(0.7, 0.15, 0.15)
):
    random.seed(42)

    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(dst, split), exist_ok=True)

    for species in os.listdir(src):
        files = [
            f for f in os.listdir(os.path.join(src, species))
            if f.lower().endswith('.jpg')
        ]

        if len(files) < target:
            continue

        files = random.sample(files, target)

        n_train = int(target * ratios[0])
        n_val = int(target * ratios[1])

        splits = {
            'train': files[:n_train],
            'val': files[n_train:n_train+n_val],
            'test': files[n_train+n_val:]
        }

        for split, imgs in splits.items():
            out_dir = os.path.join(dst, split, species)
            os.makedirs(out_dir, exist_ok=True)

            for img in imgs:
                shutil.copy2(
                    os.path.join(src, species, img),
                    os.path.join(out_dir, img)
                )


In [ ]:
df = load_and_filter_csv('snakes.csv', min_samples=200)
download_images(df)
split_dataset()


✔ Осталось видов: 26
✔ Всего изображений: 28864


Виды:  77%|███████▋  | 20/26 [1:52:27<25:44, 257.40s/it]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
train_ds = datasets.ImageFolder('dataset_final/train', transform=train_tfms)
val_ds   = datasets.ImageFolder('dataset_final/val', transform=val_tfms)
test_ds  = datasets.ImageFolder('dataset_final/test', transform=val_tfms)

num_classes = len(train_ds.classes)
print("Classes:", num_classes)


In [ ]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(num_classes),
    y=train_ds.targets
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)


In [ ]:
class SnakeClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.backbone = models.efficientnet_v2_s(
            weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
        )

        in_features = self.backbone.classifier[1].in_features

        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)


In [ ]:
model = SnakeClassifier(num_classes).to(device)


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)


In [ ]:
def train_one_epoch(model, loader):
    model.train()
    running_loss = 0

    for x, y in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        preds = model(x)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x)

            all_preds.extend(preds.argmax(1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return acc, all_labels, all_preds


In [ ]:
EPOCHS = 10
best_acc = 0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss = train_one_epoch(model, train_loader)
    val_acc, _, _ = evaluate(model, val_loader)

    print(f"Train loss: {train_loss:.4f}")
    print(f"Val accuracy: {val_acc:.4f}")

    # сохраняем лучшую модель
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_snake_model.pth')


In [ ]:
model.load_state_dict(torch.load('best_snake_model.pth'))
test_acc, y_true, y_pred = evaluate(model, test_loader)

print(f"TEST accuracy: {test_acc:.4f}")


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 10))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
torch.save({
    'model_state': model.state_dict(),
    'class_names': train_ds.classes
}, 'snake_classifier_final.pth')
